In [ ]:
# This script demonstrates a LLM Based Parallel Workflow using LangGraph.
# This workflow will evaluate an essay written by UPSC students and provide feedback on various aspects of the essay, such as (language)grammar, Clarity of thought, and Depth of analysis. 
# LLM will used to generate feedback for each aspect of the essay, and the results will be aggregated to provide a comprehensive evaluation of the essay. 
# LLM will also generate a score for each aspect of the essay, which will be used to calculate an overall/average score for the essay.
# The workflow will be executed in parallel to speed up the evaluation process. 
# An essay will be provided as input to the workflow, and the output will be a summary of the evaluation results and average score.


In [5]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import operator

/Users/schaudha/chef_progress/sushil_workspace/ai_workspace/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# load environment variables from .env file
load_dotenv()

True

In [4]:
# Define Evaluation Schema for each aspect of the essay
class EvaluationSchema(BaseModel):
    feedback: str = Field(..., description="Feedback for the aspect of the essay")
    score: int = Field(..., description="Score for the aspect of the essay", ge = 0, le = 10)

In [6]:
# Create LLM Models. We will use 2 types of models here: 1: Noraml LLM Model, 2: LLM that will output structured data based on the EvaluationSchema defined above.
llm = ChatOpenAI(model_name="gpt-4o-mini") # Normal LLM Model - gpt-4o-mini is a smaller and faster variant of GPT-4, suitable for tasks that require quick responses ans supports structured output.
structured_llm = llm.with_structured_output(EvaluationSchema) # LLM that will output structured data based on the EvaluationSchema defined above.

In [35]:
test_prompt = """India in the Age of AI
As the world enters a transformative era defined by artificial intelligence (AI), India stands at a critical juncture — one where it can either emerge as a global leader in AI innovation or risk falling behind in the technology race. The age of AI brings with it immense promise as well as unprecedented challenges, and how India navigates this landscape will shape its socio-economic and geopolitical future.

India's strengths in the AI domain are rooted in its vast pool of skilled engineers, a thriving IT industry, and a growing startup ecosystem. With over 5 million STEM graduates annually and a burgeoning base of AI researchers, India possesses the intellectual capital required to build cutting-edge AI systems. Institutions like IITs, IIITs, and IISc have begun fostering AI research, while private players such as TCS, Infosys, and Wipro are integrating AI into their global services. In 2020, the government launched the National AI Strategy (AI for All) with a focus on inclusive growth, aiming to leverage AI in healthcare, agriculture, education, and smart mobility.

One of the most promising applications of AI in India lies in agriculture, where predictive analytics can guide farmers on optimal sowing times, weather forecasts, and pest control. In healthcare, AI-powered diagnostics can help address India’s doctor-patient ratio crisis, particularly in rural areas. Educational platforms are increasingly using AI to personalize learning paths, while smart governance tools are helping improve public service delivery and fraud detection.

However, the path to AI-led growth is riddled with challenges. Chief among them is the digital divide. While metropolitan cities may embrace AI-driven solutions, rural India continues to struggle with basic internet access and digital literacy. The risk of job displacement due to automation also looms large, especially for low-skilled workers. Without effective skilling and re-skilling programs, AI could exacerbate existing socio-economic inequalities.

Another pressing concern is data privacy and ethics. As AI systems rely heavily on vast datasets, ensuring that personal data is used transparently and responsibly becomes vital. India is still shaping its data protection laws, and in the absence of a strong regulatory framework, AI systems may risk misuse or bias.

To harness AI responsibly, India must adopt a multi-stakeholder approach involving the government, academia, industry, and civil society. Policies should promote open datasets, encourage responsible innovation, and ensure ethical AI practices. There is also a need for international collaboration, particularly with countries leading in AI research, to gain strategic advantage and ensure interoperability in global systems.

India’s demographic dividend, when paired with responsible AI adoption, can unlock massive economic growth, improve governance, and uplift marginalized communities. But this vision will only materialize if AI is seen not merely as a tool for automation, but as an enabler of human-centered development.

In conclusion, India in the age of AI is a story in the making — one of opportunity, responsibility, and transformation. The decisions we make today will not just determine India’s AI trajectory, but also its future as an inclusive, equitable, and innovation-driven society.
"""

In [12]:
response = structured_llm.invoke(test_prompt)
print(response)
print("Feedback:", response.feedback)
print("Score:", response.score)

feedback='The essay is well-written, with clear language and a coherent structure. However, there are minor grammatical errors, such as "AI can either emerge as a global leader in AI innovation or risk falling behind in the technology race" where the repetition of "AI" could be avoided for more elegant phrasing. Additionally, some sentences, while technically correct, could be further polished for conciseness and flow. Overall, the language is academic and suitable for the context, with a good command of vocabulary and style. Minor issues do not detract significantly from overall clarity and communication.  ' score=8
Feedback: The essay is well-written, with clear language and a coherent structure. However, there are minor grammatical errors, such as "AI can either emerge as a global leader in AI innovation or risk falling behind in the technology race" where the repetition of "AI" could be avoided for more elegant phrasing. Additionally, some sentences, while technically correct, coul

In [13]:
# Define State of the Graph for the workflow. The workflow will have the following states: 
# 1: Evaluate the essay based on language and grammar
# 2: Evaluate the essay based on clarity of thought
# 3: Evaluate the essay based on depth of analysis.
# 4: Summerized evaluation of the essay using the above feedbacks
# 5: individiual score generated for each aspect.
# 6: Avarage score of the essay based on the individual scores generated for each aspect.

# Here we are going to use reducer function to calculate the average score of the essay based on the individual scores generated for each aspect. 
# In Parallel workflow, Nodes running parallelly use the same state key. There are high chances of update/value error if all the nodes try to write the. value in the same state key.
# So we use the reducer function to merge a new value with the existing state key instead of overwriting it. 
# LangGraph calls: reducer(current_value, new_value) → merged result.

# Without reducer - Last write wins (conflict in parallel)
# 	With reducer - Values are combined safely
# Common reducers:

# operator.add — appends lists or sums numbers if the time is int/float
# lambda a, b: max(a, b) — keeps highest value
# lambda a, b: {**a, **b} — merges dicts

class EssayState(TypedDict):
    essay_text: str
    language_grammar: str
    clarity_of_thought: str
    depth_of_analysis: str
    summarized_evaluation: str
    individual_scores: Annotated[list[int], Field(description="List of individual scores for each aspect of the essay"), operator.add]
    average_score: int

In [14]:
# Define functions to evaluate the essay based on different aspects.
# 1: Evaluate the essay based on language and grammar.
def eval_language_grammar(state: EssayState):
    # Implement the logic to evaluate language and grammar
    prompt = f"You are an expert essay evaluator for UPSC aspirants. Evaluate the following essay based on language and grammar and provide feedback and score (0-10) for the aspect. Essay: {state['essay_text']}."
    response = structured_llm.invoke(prompt)
    return {"language_grammar": response.feedback, "individual_scores": [response.score]}

In [15]:
# 2: Evaluate the essay based on clarity of thought.
def eval_clarity_of_thought(state: EssayState):
    # Implement the logic to evaluate clarity of thought
    prompt = f"You are an expert essay evaluator for UPSC aspirants. Evaluate the following essay based on clarity of thought and provide feedback and score (0-10) for the aspect. Essay: {state['essay_text']}."
    response = structured_llm.invoke(prompt)
    return {"clarity_of_thought": response.feedback, "individual_scores": [response.score]}

In [16]:
# 3: Evaluate the essay based on depth of analysis.
def eval_depth_of_analysis(state: EssayState):
    # Implement the logic to evaluate depth of analysis
    prompt = f"You are an expert essay evaluator for UPSC aspirants. Evaluate the following essay based on depth of analysis and provide feedback and score (0-10) for the aspect. Essay: {state['essay_text']}."
    response = structured_llm.invoke(prompt)
    return {"depth_of_analysis": response.feedback, "individual_scores": [response.score]}

In [17]:
# Summerize the evaluation of the essay using the feedbacks generated for each aspect.
def summarize_evaluation(state: EssayState):
    # Implement the logic to summarize the evaluation of the essay using the feedbacks generated for each aspect.
    prompt = f"You are an expert essay evaluator for UPSC aspirants. Summarize the evaluation of the following essay based on the feedbacks generated for each aspect (language and grammar, clarity of thought, depth of analysis) and provide a comprehensive evaluation. Essay: {state['essay_text']}. Feedbacks: Language and Grammar: {state['language_grammar']}, Clarity of Thought: {state['clarity_of_thought']}, Depth of Analysis: {state['depth_of_analysis']}."
    response = llm.invoke(prompt)
    
    # Calculate the Average score of the essay based on the individual scores generated for each aspect.
    average_score = sum(state['individual_scores']) / len(state['individual_scores']) if state['individual_scores'] else 0
    
    return {"summarized_evaluation": response.content, "average_score": average_score}


In [30]:
# Create a StateGraph for the workflow.
graph = StateGraph(EssayState)

In [31]:
# Add Nodes to the graph for each aspect of the essay evaluation.
graph.add_node("Evaluate Language and Grammar", eval_language_grammar)
graph.add_node("Evaluate Clarity of Thought", eval_clarity_of_thought)
graph.add_node("Evaluate Depth of Analysis", eval_depth_of_analysis)
graph.add_node("Summarize Evaluation", summarize_evaluation)

In [32]:
# Add Edges to the graph to define the flow of the workflow. The edges define the order in which the nodes will be executed.
graph.add_edge(START, "Evaluate Language and Grammar")
graph.add_edge(START, "Evaluate Clarity of Thought")
graph.add_edge(START, "Evaluate Depth of Analysis")
graph.add_edge("Evaluate Language and Grammar", "Summarize Evaluation")
graph.add_edge("Evaluate Clarity of Thought", "Summarize Evaluation")
graph.add_edge("Evaluate Depth of Analysis", "Summarize Evaluation")
graph.add_edge("Summarize Evaluation", END)
print("Graph Edges:", graph.edges)

Graph Edges: {('Evaluate Depth of Analysis', 'Summarize Evaluation'), ('Evaluate Clarity of Thought', 'Summarize Evaluation'), ('__start__', 'Evaluate Language and Grammar'), ('Summarize Evaluation', '__end__'), ('Evaluate Language and Grammar', 'Summarize Evaluation'), ('__start__', 'Evaluate Clarity of Thought'), ('__start__', 'Evaluate Depth of Analysis')}


In [34]:
# Compile the graph to create a workflow that can be executed.
compiled_graph = graph.compile()

In [39]:
essay2 = """India and AI Time

Now world change very fast because new tech call Artificial Intel… something (AI). India also want become big in this AI thing. If work hard, India can go top. But if no careful, India go back.

India have many good. We have smart student, many engine-ear, and good IT peoples. Big company like TCS, Infosys, Wipro already use AI. Government also do program “AI for All”. It want AI in farm, doctor place, school and transport.

In farm, AI help farmer know when to put seed, when rain come, how stop bug. In health, AI help doctor see sick early. In school, AI help student learn good. Government office use AI to find bad people and work fast.

But problem come also. First is many villager no have phone or internet. So AI not help them. Second, many people lose job because AI and machine do work. Poor people get more bad.

One more big problem is privacy. AI need big big data. Who take care? India still make data rule. If no strong rule, AI do bad.

India must all people together – govern, school, company and normal people. We teach AI and make sure AI not bad. Also talk to other country and learn from them.

If India use AI good way, we become strong, help poor and make better life. But if only rich use AI, and poor no get, then big bad thing happen.

So, in short, AI time in India have many hope and many danger. We must go right road. AI must help all people, not only some. Then India grow big and world say "good job India"."""

In [43]:
# Define the input
input_data = {
    "essay_text": test_prompt,
    # "essay_text": essay2
}

In [42]:
# Execute the workflow with the input data and print the output.
output = compiled_graph.invoke(input_data)
print("Output:\n", output)

Output:
 {'essay_text': 'India and AI Time\n\nNow world change very fast because new tech call Artificial Intel… something (AI). India also want become big in this AI thing. If work hard, India can go top. But if no careful, India go back.\n\nIndia have many good. We have smart student, many engine-ear, and good IT peoples. Big company like TCS, Infosys, Wipro already use AI. Government also do program “AI for All”. It want AI in farm, doctor place, school and transport.\n\nIn farm, AI help farmer know when to put seed, when rain come, how stop bug. In health, AI help doctor see sick early. In school, AI help student learn good. Government office use AI to find bad people and work fast.\n\nBut problem come also. First is many villager no have phone or internet. So AI not help them. Second, many people lose job because AI and machine do work. Poor people get more bad.\n\nOne more big problem is privacy. AI need big big data. Who take care? India still make data rule. If no strong rule, 